In [ ]:
%load_ext autoreload
%autoreload 2

# Error growth analysis for ephemeris orbit fitting
import numpy as np
import pylupnt as pnt
import matplotlib.pyplot as plt
import os

In [ ]:
basedir = "/Users/keidaiiiyama/Documents/sw_navlab/LuPNT-private/output/ephemeris/"
esim_dir = basedir + "data/ephemeris/"
orbm_dir = basedir + "data/orbits"
figdir = basedir + "figures/errorgrowth/"

if not os.path.exists(figdir):
    os.makedirs(figdir)

In [ ]:
from src.orbit_manager import OrbitManager

orbit = "Polar"
# orbit = "LCRNS"
# orbit = "Moonlight"
# orbit = "LNSS"

n_period = 2
dt = 10.0

orbm = OrbitManager(
    orbit, dyn=None, n_period=n_period, dt=dt, data_dir=orbm_dir, overwrite=False
)

In [ ]:
from src.error_propagation import run_pertubation_analysis

# parameters
start_M = np.linspace(0, 360, 4, endpoint=False)  # [deg]
# prop_mins = np.array([30, 120, 240, 360, 480])  #  120, 240, 360, 480]
prop_mins = np.array([1, 3, 5, 10, 15]) * 24 * 60  # minutes, 1, 3, 5, 10, 15 days
init_pos_errs = np.array([1, 3, 10])  # meters, 1 sigma
init_vel_errs = np.array([1, 3, 10]) * 1e-4  # mm/s, 1 sigma
n_MC = 10  # of Monte Carlo simulations

# numbers
results = run_pertubation_analysis(
    start_M, prop_mins, init_pos_errs, init_vel_errs, n_MC, orbm, n_jobs=10
)

In [ ]:
# plot results
plot_Ms = np.array([0, 90, 180, 270])  # deg

if prop_mins[-1] > 24 * 60:
    use_days = True
    prop_times = [pm / 24 / 60 for pm in prop_mins]
    scale_pos = 1e-3
    scale_v = 1
    ypos_label = "Position error [km]"
    yvel_label = "Velocity error [m/s]"
    ypos_lims = [1e-4, 100]
    yvel_lims = [1e-4, 10]
else:
    use_days = False
    prop_times = [pm for pm in prop_mins]
    scale_pos = 1
    scale_v = 1e3
    ypos_label = "Position error [m]"
    yvel_label = "Velocity error [mm/s]"
    ypos_lims = [0.1, 1000]
    yvel_lims = [0.1, 100]


maxhr = prop_mins[-1] / 60

# unpack results
error_rnorm = results["error_rnorm"]  # (n_init_pos_err, n_MC, n_M, n_prop)
error_vnorm = results["error_vnorm"]

fig, axes = plt.subplots(2, len(plot_Ms), figsize=(4 * len(plot_Ms), 6))

for i, M in enumerate(plot_Ms):
    ax = axes[0, i]
    for j, pe in enumerate(init_pos_errs):
        mean_errs = np.mean(error_rnorm[j, :, i, :] * scale_pos, axis=0)
        std_errs = np.std(error_rnorm[j, :, i, :] * scale_pos, axis=0)
        ax.plot(prop_times, mean_errs, "o-", label=f"Init err = {pe:.2f} m")
        p95_errs = np.percentile(error_rnorm[j, :, i, :] * scale_pos, 95, axis=0)
        p5_errs = np.percentile(error_rnorm[j, :, i, :] * scale_pos, 5, axis=0)
        # fill between p5 and p95
        ax.fill_between(prop_times, p5_errs, p95_errs, alpha=0.2)
        # ax.plot(results["prop_mins"], p95_errs, "--", color=ax.lines[-1].get_color(), alpha=0.5)
        # ax.plot(results["prop_mins"], p5_errs, "--", color=ax.lines[-1].get_color(), alpha=0.5)
    ax.set_title(f"Position error growth (start M={M} deg)")
    if use_days:
        ax.set_xlabel("Propagation time [days]")
    else:
        ax.set_xlabel("Propagation time [mins]")
    ax.set_ylabel(ypos_label)
    ax.set_xticks(prop_times)
    ax.set_ylim(ypos_lims)
    ax.set_yscale("log")
    ax.legend()
    ax.grid()

    ax = axes[1, i]
    for j, ve in enumerate(init_vel_errs):
        mean_errs = np.mean(error_vnorm[j, :, i, :], axis=0) * scale_v
        p95_errs = np.percentile(error_vnorm[j, :, i, :], 95, axis=0) * scale_v
        p5_errs = np.percentile(error_vnorm[j, :, i, :], 5, axis=0) * scale_v
        ax.plot(
            prop_times,
            mean_errs,
            "o-",
            label=f"Init err = {ve*1000:.2f} mm/s",
        )
        # fill between p5 and p95
        ax.fill_between(prop_times, p5_errs, p95_errs, alpha=0.2)
    ax.set_title(f"Velocity error growth (start M={M} deg)")
    if use_days:
        ax.set_xlabel("Propagation time [days]")
    else:
        ax.set_xlabel("Propagation time [mins]")
    ax.set_ylabel(yvel_label)
    ax.set_xticks(prop_times)
    ax.set_ylim(yvel_lims)
    ax.set_yscale("log")
    ax.legend()
    ax.grid()

plt.tight_layout()
plt.savefig(figdir + f"/error_growth_{orbit}_nMC{n_MC}_maxhr{maxhr:.0f}.pdf", dpi=300)
plt.show()

In [ ]:
# save results to pickle
import pickle

with open(figdir + f"/error_growth_{orbit}_nMC{n_MC}_maxhr{maxhr:.0f}.pkl", "wb") as f:
    pickle.dump(results, f)